# Image-to-3D Benchmark — BATCH 1: TripoSR + LGM
Evaluates **TripoSR** and **LGM** on the curated GSO benchmark.

In [ ]:
# --- Error logging setup ---
import sys, traceback, datetime
from pathlib import Path
LOG_FILE = Path("/kaggle/working/outputs/benchmark_log.txt")
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, data):
        for f in self.files:
            f.write(data)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()
log_f = open(LOG_FILE, "a")
sys.stdout = Tee(sys.stdout, log_f)
sys.stderr = Tee(sys.stderr, log_f)
print(f"=== BENCHMARK LOG START {datetime.datetime.now()} ===")

In [ ]:
import os, subprocess, sys, glob, gc, json
from pathlib import Path

# 1. Headless OpenGL & T4 CUDA compilation flags (sm_75 only -> 10x faster C++/CUDA builds!)
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPCONV_ALGO"] = "native"
os.environ["SPCONV_DISABLE_JIT"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MAX_JOBS"] = "4"

# 2. HuggingFace token check
_hf_token = None
try:
    for _k in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"]:
        if os.environ.get(_k):
            _hf_token = os.environ[_k]
            print(f"HF_TOKEN found in environment via {_k} ✅")
            break
    if not _hf_token:
        from kaggle_secrets import UserSecretsClient
        _client = UserSecretsClient()
        for _k in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"]:
            try:
                _val = _client.get_secret(_k)
                if _val:
                    _hf_token = _val
                    print(f"HF_TOKEN found in Kaggle Secrets under '{_k}' ✅")
                    break
            except Exception:
                pass
except Exception as _e:
    print(f"Notice: Kaggle secrets check: {type(_e).__name__}: {_e}")

if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf_token
    os.environ["HUGGINGFACE_TOKEN"] = _hf_token
    try:
        from huggingface_hub import login
        login(token=_hf_token, write_to_disk=True)
    except Exception:
        pass
    print("HF_TOKEN configured ✅")
else:
    print("Notice: HF_TOKEN not attached (public/mirror models will still work).")

import torch
print(f"torch {torch.__version__} (CUDA: {torch.version.cuda})")
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU attached! Enable Accelerator > GPU T4 x2.")
props = torch.cuda.get_device_properties(0)
print(f"CUDA OK: {props.name} | {props.total_memory/1e9:.1f} GB VRAM")

NUM_SAMPLES = 10
GSO_DIR = "/kaggle/working/data/gso"
OUT = Path("/kaggle/working/outputs"); OUT.mkdir(parents=True, exist_ok=True)

import numpy
open("/tmp/constraints.txt", "w").write(f"numpy=={numpy.__version__}\n")
print(f"Config: NUM_SAMPLES={NUM_SAMPLES} | numpy pinned: {numpy.__version__}")

In [ ]:
import os, sys, subprocess, datetime
import numpy as _np
if not os.path.exists("/tmp/constraints.txt"):
    open("/tmp/constraints.txt", "w").write(f"numpy=={_np.__version__}\n")

!pip install -q -c /tmp/constraints.txt trimesh scipy lpips rembg onnxruntime pooch opencv-python-headless pyrender pandas huggingface_hub "Pillow>=10.0"

import torch
assert torch.cuda.is_available(), "torch lost CUDA during installs"
r = subprocess.run([sys.executable, "-c", "import rembg"], capture_output=True, text=True)
if r.returncode != 0:
    !pip install -q -c /tmp/constraints.txt --force-reinstall --no-deps onnxruntime

subprocess.run([sys.executable, "-c", "import lpips; lpips.LPIPS(net='vgg')"], check=True)
from rembg import new_session
_sess = new_session("isnet-general-use")
print("=== CELL 3 GATE: PASS (LPIPS + ISNet cached) ✅ ===")

In [ ]:
import os, sys, shutil, hashlib, subprocess

CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
shutil.rmtree(CODE_DIR, ignore_errors=True)
_r = subprocess.run(
    ["git", "clone", "--depth", "1", "-b", "main", "https://github.com/anubhavaanand/image-to-3d.git", CODE_DIR],
    capture_output=True, text=True
)
if _r.returncode != 0 or not os.path.exists(os.path.join(CODE_DIR, "src")):
    raise RuntimeError("GitHub clone failed:\n" + _r.stderr[-500:])

_sha = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from src.evaluation.benchmark_runner import BenchmarkRunner, load_gso_samples, bake_texture_to_vertices
from src.engines.triposr_engine import TriposrEngine
from src.engines.lgm_engine import LGMEngine
from src.engines.trellis_engine import TrellisEngine
from src.engines.spar3d_engine import Spar3dEngine
os.makedirs("/kaggle/working/outputs", exist_ok=True)
with open("/kaggle/working/outputs/code_sha.txt", "w") as _f:
    _f.write(_sha + "\n")
print(f"code cloned fresh @ {_sha[:12]} — all 4 engines imported OK ✅ (sha recorded in outputs/code_sha.txt)")

In [ ]:
import os, sys, glob, zipfile, tempfile, shutil, random
from pathlib import Path
import numpy as np, trimesh
from PIL import Image as PILImage

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
if "GSO_DIR" not in globals():
    GSO_DIR = "/kaggle/working/data/gso"
if "NUM_SAMPLES" not in globals():
    NUM_SAMPLES = 10
if "CODE_DIR" not in globals():
    CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.evaluation.benchmark_runner import load_gso_samples, bake_texture_to_vertices, render_gso_mesh_view

CANONICAL_10_GSO = [    "ASICS_GELLinksmaster_WhiteCoffeeSand",
    "3D_Dollhouse_Happy_Brother",
    "3D_Dollhouse_Sofa",
    "Bradshaw_International_11642_7_Qt_MP_Plastic_Bowl",
    "Android_Figure_Panda",
    "Animal_Planet_Foam_2Headed_Dragon",
    "Black_and_Decker_TR3500SD_2Slice_Toaster",
    "Down_To_Earth_Orchid_Pot_Ceramic_Red",
    "Dog",
    "Dino_3",
]
if NUM_SAMPLES <= 10:
    TARGET_OBJECTS = CANONICAL_10_GSO[:NUM_SAMPLES]
else:
    _hp_list = []
    for _hp in ["/kaggle/working/handpicked.txt", os.path.join(CODE_DIR, "handpicked.txt")] + glob.glob("/kaggle/input/*/handpicked.txt"):
        if os.path.exists(_hp):
            _hp_list = [l.strip() for l in open(_hp) if l.strip() and not l.strip().startswith("#")]
            break
    _seen = set()
    TARGET_OBJECTS = []
    for _o in CANONICAL_10_GSO + _hp_list:
        if _o not in _seen:
            _seen.add(_o)
            TARGET_OBJECTS.append(_o)
        if len(TARGET_OBJECTS) >= NUM_SAMPLES:
            break
assert len(TARGET_OBJECTS) == NUM_SAMPLES, f"TARGET_OBJECTS has {len(TARGET_OBJECTS)} but NUM_SAMPLES={NUM_SAMPLES}"
assert len(set(TARGET_OBJECTS)) == len(TARGET_OBJECTS), "duplicate object in TARGET_OBJECTS"

def count_valid(root):
    root = Path(root)
    if not root.exists():
        return 0
    return sum(
        1 for d in root.iterdir()
        if d.is_dir() and (list((d/"meshes").glob("*.obj")) or list((d/"meshes").glob("*.ply")))
    )

REPO = "suvadityamuk/google-scanned-objects-raw"
def have_all(root):
    root = Path(root)
    for n in TARGET_OBJECTS:
        m = root / n / "meshes"
        if not (m.is_dir() and (list(m.glob("*.obj")) or list(m.glob("*.ply")))):
            return False
    return True
if not have_all(GSO_DIR):
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi()
    all_zips = sorted(f for f in api.list_repo_files(REPO, repo_type="dataset") if f.endswith(".zip"))
    _zset = set(all_zips)
    _missing = [n for n in TARGET_OBJECTS if (n + ".zip") not in _zset]
    if _missing:
        raise RuntimeError(f"GSO zips missing upstream: {_missing}")
    wanted = [n + ".zip" for n in TARGET_OBJECTS]

    def ingest(zip_name):
        obj_id = zip_name[:-4]
        try:
            zp = hf_hub_download(REPO, zip_name, repo_type="dataset", local_dir="/kaggle/working/gso_zips")
            with tempfile.TemporaryDirectory() as td:
                with zipfile.ZipFile(zp) as z:
                    z.extractall(td)
                hits = list(Path(td).rglob("*.obj")) + list(Path(td).rglob("*.ply"))
                if not hits:
                    return False
                mdir = Path(GSO_DIR) / obj_id / "meshes"
                mdir.mkdir(parents=True, exist_ok=True)
                for h in hits:
                    shutil.copy(h, mdir / h.name)
                for sib in hits[0].parent.iterdir():
                    if sib.suffix.lower() in (".mtl", ".png", ".jpg", ".jpeg"):
                        shutil.copy(sib, mdir / sib.name)
                for tex in Path(td).rglob("*.png"):
                    if not (mdir / tex.name).exists():
                        shutil.copy(tex, mdir / tex.name)
            return True
        except Exception as e:
            print("  skip", obj_id, str(e)[:80])
            return False

    ok = 0
    target_n = len(TARGET_OBJECTS)
    for zname in wanted:
        if ingest(zname):
            ok += 1
        if ok >= target_n:
            break
    print(f"ingested {ok} objects into {GSO_DIR}")

rendered = 0
for d in sorted(Path(GSO_DIR).iterdir()):
    if not d.is_dir():
        continue
    mcands = list((d/"meshes").glob("*.obj")) + list((d/"meshes").glob("*.ply"))
    if not mcands or (d/"images"/".upright_v6").exists():
        continue
    try:
        (d/"images").mkdir(parents=True, exist_ok=True)
        # Render 000.png (1024x1024 RGBA) and 001.png (256x256 RGB) directly from the native
        # 4096x4096 GSO UV texture map (no vertex-baking bottleneck or double-gamma washout).
        rgba_000 = render_gso_mesh_view(mcands[0], azimuth_deg=20.0, elevation_deg=15.0, resolution=1024, return_rgba=True)
        rgb_001 = render_gso_mesh_view(mcands[0], azimuth_deg=65.0, elevation_deg=15.0, resolution=256, return_rgba=False)
        if rgba_000 is None or rgb_001 is None:
            continue
        PILImage.fromarray(rgba_000, mode="RGBA").save(d/"images"/"000.png")
        PILImage.fromarray(rgb_001).save(d/"images"/"001.png")
        (d/"images"/".upright_v6").write_text("ok")
        rendered += 1
    except Exception as e:
        print("render failed:", d.name, str(e)[:100])

samples = load_gso_samples(GSO_DIR, num_samples=NUM_SAMPLES, object_ids=TARGET_OBJECTS)
_missing = [o for o in TARGET_OBJECTS if o not in {sm.sample_id for sm in samples}]
if _missing or len(samples) != len(TARGET_OBJECTS):
    raise RuntimeError(f"GSO pre-flight failed: {len(samples)}/{len(TARGET_OBJECTS)} ready, missing={_missing}")
print(f"GSO ready: rendered {rendered} upright RGBA views | loaded {len(samples)}/{len(TARGET_OBJECTS)} canonical samples ✅")

In [ ]:
# --- TripoSR (Triplane NeRF -> 256^3 Marching Cubes) ---
import os, sys, gc, datetime, pathlib, subprocess, traceback
import torch

try:
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    if "CODE_DIR" not in globals():
        CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
    if str(CODE_DIR) not in sys.path:
        sys.path.insert(0, str(CODE_DIR))

    if not os.path.isdir("/kaggle/working/TripoSR"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/VAST-AI-Research/TripoSR.git", "/kaggle/working/TripoSR"], check=True)
    if "/kaggle/working/TripoSR" not in sys.path:
        sys.path.insert(0, "/kaggle/working/TripoSR")

    !pip install -q -c /tmp/constraints.txt omegaconf einops pytorch_lightning e3nn transformers

    pathlib.Path("/kaggle/working/torchmcubes.py").write_text("""import numpy as np
import torch
from skimage import measure

def marching_cubes(level, iso=0.0):
    if level.dim() == 4:
        level = level.squeeze(0)
    vol = level.detach().cpu().numpy().astype(np.float32)
    try:
        v, f, _, _ = measure.marching_cubes(vol, float(iso))
        v = np.ascontiguousarray(v[:, [2, 1, 0]], dtype=np.float32)
        f = np.ascontiguousarray(f, dtype=np.int64)
    except Exception:
        return (torch.zeros((0, 3), dtype=torch.float32),
                torch.zeros((0, 3), dtype=torch.int64))
    return torch.from_numpy(v), torch.from_numpy(f)
""")
    if "/kaggle/working" not in sys.path:
        sys.path.insert(0, "/kaggle/working")

    from src.engines.triposr_engine import TriposrEngine
    from src.evaluation.benchmark_runner import BenchmarkRunner

    tr_engine = TriposrEngine(device="cuda")
    tr_engine.load_model()
    runner = BenchmarkRunner(dataset_path=GSO_DIR, engines=[tr_engine], num_samples=NUM_SAMPLES, object_ids=TARGET_OBJECTS, output_dir="/kaggle/working/outputs/triposr")
    try:
        print(f"[TripoSR] {datetime.datetime.now().strftime('%H:%M:%S')} Starting run...")
        runner.run()
        print(f"[TripoSR] {datetime.datetime.now().strftime('%H:%M:%S')} COMPLETE ✅")
        runner.save_results("/kaggle/working/outputs/triposr_results.json")
        display(runner.get_summary_df())
    finally:
        del runner, tr_engine
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception as e:
    print(f"[TripoSR] Error: {type(e).__name__}: {e}")
    traceback.print_exc()

In [ ]:
# --- LGM (ImageDream 4-View + Asymmetric U-Net -> 65,536 3D Gaussians -> 256^3 MC Mesh) ---
import os, sys, gc, glob, datetime, subprocess, traceback
import torch

try:
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
    os.environ["MAX_JOBS"] = "4"
    if "CODE_DIR" not in globals():
        CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
    if str(CODE_DIR) not in sys.path:
        sys.path.insert(0, str(CODE_DIR))

    !pip install -q -c /tmp/constraints.txt --no-deps "kiui==0.3.4"
    !pip install -q -c /tmp/constraints.txt varname lazy_loader objprint rich pyyaml questionary tyro einops roma plyfile

    for _h in glob.glob("/usr/local/lib/python*/dist-packages/kiui/typing.py"):
        open(_h, "w").write("from typing import Sequence, List, Tuple, Dict, Any, Optional, Union, Literal, Callable\nfrom torch import Tensor\nfrom numpy import ndarray\n")
    for _k in [k for k in sys.modules if k.startswith("kiui")]:
        del sys.modules[_k]

    for _url, _dest in [
        ("https://github.com/ashawkey/mvdream_diffusers.git", "/kaggle/working/mvdream"),
        ("https://github.com/3DTopia/LGM.git", "/kaggle/working/LGM"),
        ("https://github.com/ashawkey/diff-gaussian-rasterization.git", "/kaggle/working/dgr"),
    ]:
        if not os.path.isdir(_dest):
            subprocess.run(["git", "clone", "--recursive", "--depth", "1", _url, _dest], check=True)
    !pip install -q -c /tmp/constraints.txt --no-build-isolation /kaggle/working/dgr

    for _p in ["/kaggle/working/mvdream", "/kaggle/working/LGM"]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    from src.engines.lgm_engine import LGMEngine
    from src.evaluation.benchmark_runner import BenchmarkRunner

    lg_engine = LGMEngine(device="cuda")
    lg_engine.load_model(big=True)
    runner = BenchmarkRunner(dataset_path=GSO_DIR, engines=[lg_engine], num_samples=NUM_SAMPLES, object_ids=TARGET_OBJECTS, output_dir="/kaggle/working/outputs/lgm")
    try:
        print(f"[LGM] {datetime.datetime.now().strftime('%H:%M:%S')} Starting run...")
        runner.run()
        print(f"[LGM] {datetime.datetime.now().strftime('%H:%M:%S')} COMPLETE ✅")
        runner.save_results("/kaggle/working/outputs/lgm_results.json")
        display(runner.get_summary_df())
    finally:
        del runner, lg_engine
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception as e:
    print(f"[LGM] Error: {type(e).__name__}: {e}")
    traceback.print_exc()

In [ ]:
import glob, json
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for p in sorted(glob.glob("/kaggle/working/outputs/*_results.json")):
    try:
        rows += json.load(open(p))
    except Exception as e:
        print("skip", p, str(e)[:80])

df = pd.DataFrame(rows)
_before = len(df)
df = df.drop_duplicates(subset=["engine", "sample_id"], keep="last")
if len(df) != _before:
    print(f"\u26a0 dropped {_before - len(df)} duplicate (engine, sample_id) rows")
_counts = df.groupby("engine").size().to_dict()
_bad = {e: int(c) for e, c in _counts.items() if c != NUM_SAMPLES}
if _bad:
    print(f"\u26a0 row-count mismatch vs NUM_SAMPLES={NUM_SAMPLES}: {_bad} — results incomplete!")
df.to_csv("/kaggle/working/outputs/unified_results.csv", index=False)

num_cols = ["latency_s", "vram_peak_gb", "chamfer_distance", "f_score_0.1", "f_score_0.2", "f_score_0.5", "psnr", "lpips"]
cols = [c for c in num_cols if c in df.columns]
if cols and not df.empty:
    summary = df.groupby("engine")[cols].agg(["mean", "std"]).round(4)
    print("=== UNIFIED BENCHMARK RESULTS (mean +/- std) ===")
    print(summary.to_string())
    with open("/kaggle/working/outputs/unified_table.tex", "w") as f:
        f.write(summary.reset_index().to_latex(index=False, na_rep="--"))

    colors = {"TripoSR": "#1f77b4", "LGM": "#ff7f0e", "TRELLIS": "#2ca02c", "SPAR3D": "#9467bd"}
    means = df.groupby("engine")[cols].mean()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for eng, row in means.iterrows():
        c = colors.get(eng, "grey")
        if "f_score_0.1" in row and not pd.isna(row.get("f_score_0.1")):
            axes[0].scatter(row["latency_s"], row["f_score_0.1"], s=180, c=c, label=eng, zorder=5)
        if "chamfer_distance" in row and not pd.isna(row.get("chamfer_distance")):
            axes[1].scatter(row["latency_s"], row["chamfer_distance"], s=180, c=c, label=eng, zorder=5)
    axes[0].set_xlabel("Latency (s)"); axes[0].set_ylabel("F-Score @ 0.1"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].set_xlabel("Latency (s)"); axes[1].set_ylabel("Chamfer Distance"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("/kaggle/working/outputs/pareto_frontier.png", dpi=150)
    print("saved: unified_results.csv, unified_table.tex, pareto_frontier.png ✅")

In [ ]:
import shutil, os
from pathlib import Path

WORK = Path("/kaggle/working")
KEEP = {"outputs"}
if WORK.exists():
    for item in sorted(WORK.iterdir()):
        if item.name in KEEP:
            continue
        try:
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()
        except Exception:
            pass
    print("saved outputs:", sorted(p.name for p in (WORK / "outputs").iterdir()))